## Objectives for this Notebook    
* How to use the Keras library to build a regression model
* Download and clean the data set
* Build a neural network
* Train and test the network     



In [1]:
import keras

<h2>Table of Contents</h2>


<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 4>
1. <a href="#Download-and-Clean-the-Data-Set">Download and Clean the Data Set</a><br>
2. <a href="#Import-Keras-Packages">Import Keras Packages</a><br>
3. <a href="#Build-a-Neural-Network">Build a Neural Network</a><br>
4. <a href="#Train-and-Test-the-Network">Train and Test the Network</a><br>  

</font>
</div>


In [2]:
import pandas as pd
import numpy as np 
import keras 

import warnings 
warnings.simplefilter("ignore",FutureWarning)

<strong>The dataset is about the compressive strength of different samples of concrete based on the volumes of the different ingredients that were used to make them. Ingredients include:</strong>

* Cement
* Blast furnace slag
* Fly ash
* Water
* Superplasticizer
* Coarse aggregate
* Fine aggregate

In [3]:
filepath='https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

concrete_data.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


So the first concrete sample has 540 cubic meter of cement, 0 cubic meter of blast furnace slag, 0 cubic meter of fly ash, 162 cubic meter of water, 2.5 cubic meter of superplaticizer, 1040 cubic meter of coarse aggregate, 676 cubic meter of fine aggregate. Such a concrete mix which is 28 days old, has a compressive strength of 79.99 MPa. 


In [4]:
concrete_data.shape

(1030, 9)

So, there are approximately 1000 samples to train our model on. Because of the few samples, we have to be careful not to overfit the training data.


In [6]:
concrete_data.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [8]:
concrete_data.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

The data looks pretty clean and is ready to be used to build our Model

#### Split data into predictors and target


The target variable in this problem is the concrete sample strength. Therefore, our predictors will be all the other columns.


In [9]:
concrete_data_columns = concrete_data.columns

In [11]:
predictors=concrete_data[concrete_data_columns[concrete_data_columns != 'Strength' ]]
target = concrete_data['Strength']

Let's do a quick sanity check of the predictors and the target dataframes

In [12]:
predictors.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [13]:
target.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Strength, dtype: float64

Finally, the last step is to normalize the data by substracting the mean and dividing by the standard deviation.

In [14]:
predictors_norm=(predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,0.862735,-1.217079,-0.279597
1,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,1.055651,-1.217079,-0.279597
2,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,3.551340
3,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,5.055221
4,-0.790075,0.678079,-0.846733,0.488555,-1.038638,0.070492,0.647569,4.976069


Finally, the last step is to normalize the data by substracting the mean and dividing by the standard deviation.

In [15]:
n_cols=predictors_norm.shape[1] #number of predictors

In [16]:
from keras.models import Sequential
from keras.layers import Dense,Input


## Build a Neural Network

Let's define a function that defines our regression model for us so that we can conveniently call it to create our model.


In [20]:
def regression_model():

    #create model
    model=Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(1))
    
    model.compile(optimizer='adam',loss='mean_squared_error')
    return model

The above function create a model that has two hidden layers, each of 50 hidden units.


## Train and Test the Network


Let's call the function now to create our model.

In [21]:
#build the model
model=regression_model()

In [22]:
model.fit(predictors_norm ,target,validation_split=0.3,epochs=100,verbose=2)

Epoch 1/100
23/23 - 2s - 77ms/step - loss: 1634.3572 - val_loss: 1124.3877
Epoch 2/100
23/23 - 0s - 7ms/step - loss: 1478.1246 - val_loss: 985.1884
Epoch 3/100
23/23 - 0s - 8ms/step - loss: 1233.0570 - val_loss: 783.3630
Epoch 4/100
23/23 - 0s - 7ms/step - loss: 889.6536 - val_loss: 545.2184
Epoch 5/100
23/23 - 0s - 7ms/step - loss: 538.1878 - val_loss: 347.9916
Epoch 6/100
23/23 - 0s - 7ms/step - loss: 312.5489 - val_loss: 249.5509
Epoch 7/100
23/23 - 0s - 7ms/step - loss: 243.3046 - val_loss: 214.7348
Epoch 8/100
23/23 - 0s - 7ms/step - loss: 223.8595 - val_loss: 197.2361
Epoch 9/100
23/23 - 0s - 7ms/step - loss: 210.8650 - val_loss: 187.6098
Epoch 10/100
23/23 - 0s - 7ms/step - loss: 200.1868 - val_loss: 185.4525
Epoch 11/100
23/23 - 0s - 7ms/step - loss: 191.7059 - val_loss: 179.4717
Epoch 12/100
23/23 - 0s - 8ms/step - loss: 184.8522 - val_loss: 174.8964
Epoch 13/100
23/23 - 0s - 8ms/step - loss: 179.3370 - val_loss: 171.6626
Epoch 14/100
23/23 - 0s - 7ms/step - loss: 174.3931 - v